# 04 — Customer Segmentation: RFM Methodology & Prototype (Issue #5)

This notebook is limited to the scope of **Issue #5**:

- validate the agreed RFM definitions and reference-date rule;
- demonstrate R/F/M scoring and rule-based segmentation;
- test the required edge cases on deterministic sample data;
- validate the **designed schema** of `customer_segments.csv`;
- prepare reusable code that can later run on `cleaned_retail.csv`.

> **Out of scope for Issue #5:** running the official full-data analysis and exporting the production `customer_segments.csv`.

**Core rules**

- Recency = number of calendar days since the latest valid purchase.
- Frequency = number of distinct valid invoices (`nunique(InvoiceNo)`).
- Monetary = sum of valid `Quantity × UnitPrice`.
- Reference date = one calendar day after the latest valid identified-customer purchase.

In [1]:
from pathlib import Path
import sys
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Locate repository root whether Jupyter starts in repo root or notebooks/.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing src/.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    audit_rfm_input,
    calculate_rfm,
    prepare_rfm_transactions,
    score_rfm_metric,
)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /mnt/data/issue5_notebook_revised


## 1. Deterministic sample for the prototype

The prototype intentionally uses a small synthetic transaction table rather than the official cleaned dataset.  
The sample contains every special case requested by Issue #5:

- one customer with multiple invoices;
- multiple product lines on the same invoice;
- one-order customers;
- cancelled invoice;
- missing `CustomerID`;
- anonymous `Guest`;
- non-positive quantity;
- non-positive unit price;
- exact duplicate rows;
- customers with equal RFM values.

This makes the expected results easy to verify manually.

In [2]:
sample = pd.DataFrame([
    # Customer 1: two product lines on invoice 10001 + another invoice 10002.
    ["10001", "A", 1, "2011-12-01 10:00", 10.0, "1"],
    ["10001", "B", 2, "2011-12-01 10:00",  5.0, "1"],
    ["10002", "C", 1, "2011-12-05 10:00", 20.0, "1"],

    # Customer 2: one valid purchase on the final valid purchase date.
    ["10003", "A", 1, "2011-12-09 09:00", 10.0, "2"],

    # Customers 3 and 4: deliberately identical RFM values.
    ["10004", "A", 2, "2011-12-07 09:00", 10.0, "3"],
    ["10005", "A", 2, "2011-12-07 09:00", 10.0, "4"],

    # Rows that must be excluded from the RFM population.
    ["C10006", "A", -1, "2011-12-08 09:00", 10.0, "5"],  # cancelled
    ["10007", "A", 1, "2011-12-08 09:00", 10.0, None],   # missing CustomerID
    ["10008", "A", 1, "2011-12-08 09:00", 10.0, "Guest"],# anonymous customer
    ["10009", "A", 0, "2011-12-08 09:00", 10.0, "6"],    # invalid quantity
    ["10010", "A", 1, "2011-12-08 09:00", 0.0, "7"],     # invalid price

    # Exact duplicate pair: it must contribute only once.
    ["10011", "D", 1, "2011-12-06 09:00", 15.0, "8"],
    ["10011", "D", 1, "2011-12-06 09:00", 15.0, "8"],
], columns=[
    "InvoiceNo", "StockCode", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID"
])

sample

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,CustomerID
0,10001,A,1,2011-12-01 10:00,10.00,1
1,10001,B,2,2011-12-01 10:00,5.00,1
2,10002,C,1,2011-12-05 10:00,20.00,1
3,10003,A,1,2011-12-09 09:00,10.00,2
4,10004,A,2,2011-12-07 09:00,10.00,3
5,10005,A,2,2011-12-07 09:00,10.00,4
6,C10006,A,-1,2011-12-08 09:00,10.00,5
7,10007,A,1,2011-12-08 09:00,10.00,None
8,10008,A,1,2011-12-08 09:00,10.00,Guest
9,10009,A,0,2011-12-08 09:00,10.00,6


## 2. Audit and valid-purchase population

Before customer aggregation, RFM applies the agreed eligibility rules.  
The audit is descriptive; counts can overlap because the same row may violate more than one rule.

In [3]:
audit = audit_rfm_input(sample)
pd.Series(audit, name="rows")

input_rows                     13
cancelled_rows                  1
missing_customer_rows           1
anonymous_customer_rows         1
nonpositive_quantity_rows       2
nonpositive_unit_price_rows     1
invalid_invoice_date_rows       0
exact_duplicate_rows            1
Name: rows, dtype: int64

In [4]:
valid_sample = prepare_rfm_transactions(sample)

valid_sample[
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate",
     "UnitPrice", "CustomerID", "Revenue"]
].sort_values(["CustomerID", "InvoiceDate", "InvoiceNo"])

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,CustomerID,Revenue
0,10001,A,1,2011-12-01 10:00:00,10.00,1,10.00
1,10001,B,2,2011-12-01 10:00:00,5.00,1,10.00
2,10002,C,1,2011-12-05 10:00:00,20.00,1,20.00
3,10003,A,1,2011-12-09 09:00:00,10.00,2,10.00
4,10004,A,2,2011-12-07 09:00:00,10.00,3,20.00
5,10005,A,2,2011-12-07 09:00:00,10.00,4,20.00
11,10011,D,1,2011-12-06 09:00:00,15.00,8,15.00


## 3. Calculate prototype RFM

`calculate_rfm()` derives the reference date automatically from valid purchases.

For this sample, the final valid purchase occurs on **2011-12-09**, therefore:

`ReferenceDate = 2011-12-10`

A customer who purchased on 2011-12-09 consequently has `Recency = 1`.

In [5]:
prototype_rfm, metadata = calculate_rfm(sample, return_metadata=True)

summary = pd.Series({
    "valid_transaction_rows": metadata["valid_transaction_rows"],
    "customers": metadata["customers"],
    "invoices": metadata["invoices"],
    "last_valid_purchase": metadata["last_valid_purchase"],
    "reference_date": metadata["reference_date"],
})
summary

valid_transaction_rows                      7
customers                                   5
invoices                                    6
last_valid_purchase       2011-12-09 09:00:00
reference_date            2011-12-10 00:00:00
dtype: object

In [6]:
prototype_rfm.sort_values("CustomerID")

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score,RFM_total,Segment,LastPurchaseDate,ReferenceDate
0,1,5,2,40.00,1,5,5,155,11,At Risk,2011-12-05,2011-12-10
3,2,1,1,10.00,5,1,1,511,7,New Customers,2011-12-09,2011-12-10
1,3,3,1,20.00,4,1,3,413,8,New Customers,2011-12-07,2011-12-10
2,4,3,1,20.00,4,1,3,413,8,New Customers,2011-12-07,2011-12-10
4,8,4,1,15.00,2,1,2,212,5,Hibernating,2011-12-06,2011-12-10


## 4. Verify RFM definitions and required edge cases

The checks below directly map to the Issue #5 prototype requirements.

In [7]:
c1 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("1")].iloc[0]
c2 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("2")].iloc[0]
c3 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("3")].iloc[0]
c4 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("4")].iloc[0]
c8 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("8")].iloc[0]

# Reference date: one day after the latest valid purchase.
assert metadata["reference_date"] == pd.Timestamp("2011-12-10")

# Multiple product rows on one invoice must not inflate Frequency.
# Customer 1 has invoice 10001 (2 lines) and 10002 (1 line) => Frequency = 2.
assert c1["Frequency"] == 2

# Monetary is sum(Quantity * UnitPrice):
# 1*10 + 2*5 + 1*20 = 40.
assert c1["Monetary"] == 40.0

# Customer 2 purchased on 2011-12-09; reference date is 2011-12-10.
assert c2["Recency"] == 1

# Cancelled, missing/anonymous and invalid rows are excluded.
assert set(prototype_rfm["CustomerID"]) == {"1", "2", "3", "4", "8"}

# Exact duplicate contributes once.
assert c8["Frequency"] == 1
assert c8["Monetary"] == 15.0

# Equal RFM raw values must receive equal R/F/M scores.
assert tuple(c3[["Recency", "Frequency", "Monetary"]]) == tuple(
    c4[["Recency", "Frequency", "Monetary"]]
)
assert tuple(c3[["R_score", "F_score", "M_score"]]) == tuple(
    c4[["R_score", "F_score", "M_score"]]
)

print("RFM definition and edge-case checks passed.")

RFM definition and edge-case checks passed.


## 5. Verify scoring behavior

The implementation uses quintile cutoffs (`q20`, `q40`, `q60`, `q80`) when the cutoffs are distinct.

If quantile cutoffs collapse because many values are tied or there are too few distinct values, the scorer uses a deterministic dense-value fallback. Equal raw values always receive equal scores.

Score direction:

- lower Recency → higher `R_score`;
- higher Frequency → higher `F_score`;
- higher Monetary → higher `M_score`.

In [8]:
# Low-cardinality / tied Frequency values trigger the deterministic fallback.
tie_demo = pd.Series([1, 1, 1, 2, 2, 3], name="Frequency")
tie_scores, tie_meta = score_rfm_metric(tie_demo, higher_is_better=True)

tie_result = pd.DataFrame({
    "Frequency": tie_demo,
    "F_score": tie_scores,
})
display(tie_result)
pd.Series(tie_meta)

,Frequency,F_score
0,1,1
1,1,1
2,1,1
3,2,3
4,2,3
5,3,5


method              dense_fallback
q20                           1.00
q40                           1.00
q60                           2.00
q80                           2.00
n_unique                         3
higher_is_better              True
dtype: object

In [9]:
assert tie_scores.between(1, 5).all()
assert tie_scores[tie_demo.eq(1)].nunique() == 1
assert tie_scores[tie_demo.eq(2)].nunique() == 1
assert tie_meta["method"] == "dense_fallback"

# Recency direction check: lower Recency must produce a higher/equal score.
recency_demo = pd.Series([1, 10, 30, 60, 120, 300], name="Recency")
r_scores, _ = score_rfm_metric(recency_demo, higher_is_better=False)

assert r_scores.iloc[0] > r_scores.iloc[-1]

print("Scoring direction and tie-handling checks passed.")

Scoring direction and tie-handling checks passed.


## 6. Validate the designed `customer_segments.csv` schema

Issue #5 requires the **design** of the future `customer_segments.csv`; it does not require exporting the official file.

The prototype DataFrame must therefore match the agreed customer-level schema exactly.

In [10]:
CUSTOMER_SEGMENTS_COLUMNS = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary",
    "R_score",
    "F_score",
    "M_score",
    "RFM_score",
    "RFM_total",
    "Segment",
    "LastPurchaseDate",
    "ReferenceDate",
]

assert prototype_rfm.columns.tolist() == CUSTOMER_SEGMENTS_COLUMNS
assert prototype_rfm["CustomerID"].is_unique
assert prototype_rfm["CustomerID"].notna().all()

assert prototype_rfm["Recency"].ge(1).all()
assert prototype_rfm["Frequency"].ge(1).all()
assert prototype_rfm["Monetary"].gt(0).all()

assert prototype_rfm["R_score"].between(1, 5).all()
assert prototype_rfm["F_score"].between(1, 5).all()
assert prototype_rfm["M_score"].between(1, 5).all()
assert prototype_rfm["RFM_total"].between(3, 15).all()

assert prototype_rfm["RFM_score"].astype(str).str.fullmatch(r"[1-5]{3}").all()
assert prototype_rfm["ReferenceDate"].nunique() == 1

print("Designed customer_segments.csv schema validation passed.")

Designed customer_segments.csv schema validation passed.


## 7. Production readiness for `cleaned_retail.csv`

Issue #5 only requires the code to be **ready** to run on `cleaned_retail.csv`.  
The prototype does not execute the official full-data analysis and does not export `customer_segments.csv`.

The helper below shows the intended later usage. It is defined here but deliberately not called in this prototype.

In [11]:
def run_rfm_on_cleaned_csv(csv_path: str | Path):
    """Production-readiness example for a later analysis/integration step."""
    cleaned = pd.read_csv(
        csv_path,
        dtype={"InvoiceNo": "string", "CustomerID": "string"},
        low_memory=False,
    )
    return calculate_rfm(cleaned, return_metadata=True)

# Later official-analysis usage:
#
# customer_segments, production_metadata = run_rfm_on_cleaned_csv(
#     PROJECT_ROOT / "data" / "processed" / "cleaned_retail.csv"
# )
#
# No to_csv(...) call is made in Issue #5.

## 8. Issue #5 acceptance checklist

- [x] RFM formulas and reference-date rule are explicit.
- [x] Cancelled invoices, missing/anonymous `CustomerID`, invalid values and duplicates have explicit rules.
- [x] R/F/M score direction is defined on a 1–5 scale.
- [x] Quantile ties / low-cardinality values have a deterministic fallback.
- [x] Segment labels and precedence thresholds are implemented in the reusable RFM module.
- [x] Prototype runs on deterministic sample data.
- [x] Prototype validates one/multiple orders, multi-line invoices and equal RFM values.
- [x] The future `customer_segments.csv` structure is designed and programmatically validated.
- [x] Reusable code is prepared to run later on `cleaned_retail.csv`.
- [x] The notebook does **not** export the official `customer_segments.csv`, because that is outside Issue #5 scope.